# Gene Regulatory Network using SCENIC
Author: Ian K. Hsu   
Last update: 31/07/2024   

To investigate the gene regulatory network (GRN) specific to the dataset, we initially employed SCENIC v0.12, a package designed for reconstructing the GRNs from snRNA-seq data, using the raw RNA count matrix extracted from the Seurat object. Only genes with a total expression greater than zero were retained for GRN inference.


# 1. Preprocessing

In [ ]:
%%R

library(Seurat)
library(SCopeLoomR)
library(SCENIC)

# load Seurat object (Tp or Tc). Here, we use Seurat object Tp as example
object = readRDS('../data/Tp.rds')

# count matric from RNA assay
DefaultAssay(object) <- "RNA"
exprMat <- object@assays[["RNA"]]@counts

# filtering genes with sum of expression >0
loci1 <- which(rowSums(exprMat) > 0)
exprMat_filter <- exprMat[loci1, ]

# cell information from Seurat object (meta.data)
cellInfo <- object@meta.data

loom <- build_loom("../otp/Tp.loom", dgem=exprMat_filter)
loom <- add_cell_annotation(loom, cellInfo)
close_loom(loom)

# 2. Building GRN using pySCENIC

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import loompy as lp
from MulticoreTSNE import MulticoreTSNE as TSNE
import glob


# ranking databases
f_db_glob = "../SCENIC/human/hg38_v2/*feather"
f_db_names = ' '.join( glob.glob(f_db_glob) )
# motif databases
f_motif_path = "../SCENIC/human/hg38_v2/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"

rna_loom_ifn = '../data/Tp.loom'
tf_list = '../SCENIC/human/allTFs_hg38.txt'
adj_ofn = '../otp/Tp_grnboost2_adj.csv'
reg_ofn = '../otp/Tp_reg.csv'
scenic_ofn = '../otp/Tp_scenic.loom'

In [ ]:
%%bash
python arboreto_with_multiprocessing.py \
     {rna_loom_ifn} \
     {tf_list} \
     -o {adj_ofn} \
     --num_workers 40

In [17]:
%%bash
pyscenic ctx {adj_ofn} \
    {f_db_names} \
    --annotations_fname {f_motif_path} \
    --expression_mtx_fname {rna_loom_ifn} \
    --output {reg_ofn} \
    --num_workers 40 \
    --mask_dropouts 


2024-07-31 11:49:49,968 - pyscenic.cli.pyscenic - INFO - Creating modules.

2024-07-31 11:49:51,250 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2024-07-31 11:49:55,739 - pyscenic.utils - INFO - Calculating Pearson correlations.

2024-07-31 11:49:56,293 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [True].

2024-07-31 11:50:11,398 - pyscenic.utils - INFO - Creating modules.

2024-07-31 11:52:51,452 - pyscenic.cli.pyscenic - IN


2024-07-31 11:53:03,677 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(9): database loaded in memory.

2024-07-31 11:53:03,677 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(9): database loaded in memory.

2024-07-31 11:53:03,716 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(7): database loaded in memory.

2024-07-31 11:53:03,716 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(7): database loaded in memory.

2024-07-31 11:53:03,721 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(15): database loaded in memory.

2024-07-31 11:53:03,721 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(15): database loaded in memory.

2024-07-31 11:53:03,781 - pyscenic.prune - INFO - Worke


2024-07-31 11:53:06,288 - pyscenic.prune - INFO - Worker hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings(7): motif annotations loaded in memory.

2024-07-31 11:53:06,288 - pyscenic.prune - INFO - Worker hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings(7): motif annotations loaded in memory.

2024-07-31 11:53:06,291 - pyscenic.prune - INFO - Worker hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings(2): motif annotations loaded in memory.

2024-07-31 11:53:06,291 - pyscenic.prune - INFO - Worker hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings(2): motif annotations loaded in memory.

2024-07-31 11:53:06,301 - pyscenic.prune - INFO - Worker hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings(14): motif annotations loaded in memory.

2024-07-31 11:53:06,301 - pyscenic.prune - INFO - Worker hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings(14): motif annotations loaded in memory.

2


2024-07-31 11:53:24,019 - pyscenic.transform - WARNING - Less than 80% of the genes in ASPSCR1 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:24,176 - pyscenic.transform - WARNING - Less than 80% of the genes in TCF7 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:24,302 - pyscenic.transform - WARNING - Less than 80% of the genes in ZNF561 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:24,316 - pyscenic.transform - WARNING - Less than 80% of the genes in DMAP1 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:24,369 - pyscenic.transform - WARNING - Less than 80% of the genes in DPF2 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. S


2024-07-31 11:53:35,232 - pyscenic.transform - WARNING - Less than 80% of the genes in DPF2 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:35,656 - pyscenic.transform - WARNING - Less than 80% of the genes in ZNF567 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:35,770 - pyscenic.transform - WARNING - Less than 80% of the genes in TCF12 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:36,000 - pyscenic.transform - WARNING - Less than 80% of the genes in ZNF567 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:36,484 - pyscenic.transform - WARNING - Less than 80% of the genes in TGIF1 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. S


2024-07-31 11:53:48,998 - pyscenic.transform - WARNING - Less than 80% of the genes in ZNF574 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:49,056 - pyscenic.transform - WARNING - Less than 80% of the genes in BAX could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:49,323 - pyscenic.transform - WARNING - Less than 80% of the genes in PUM3 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:49,482 - pyscenic.transform - WARNING - Less than 80% of the genes in TIMM44 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 11:53:49,851 - pyscenic.transform - WARNING - Less than 80% of the genes in BBX could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipp

... (71 outputs left out)


2024-07-31 12:07:48,233 - pyscenic.transform - WARNING - Less than 80% of the genes in SNRPB2 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:07:48,335 - pyscenic.transform - WARNING - Less than 80% of the genes in FOS could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:07:48,700 - pyscenic.transform - WARNING - Less than 80% of the genes in SOCS4 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:07:49,159 - pyscenic.transform - WARNING - Less than 80% of the genes in NFE2L2 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:07:49,180 - pyscenic.transform - WARNING - Less than 80% of the genes in SOD1 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Ski


2024-07-31 12:08:08,934 - pyscenic.transform - WARNING - Less than 80% of the genes in NR1H2 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:08:09,376 - pyscenic.transform - WARNING - Less than 80% of the genes in STAU2 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:08:09,425 - pyscenic.transform - WARNING - Less than 80% of the genes in NR1H3 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:08:09,858 - pyscenic.transform - WARNING - Less than 80% of the genes in STK40 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:08:10,335 - pyscenic.transform - WARNING - Less than 80% of the genes in STUB1 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Sk


2024-07-31 12:08:28,816 - pyscenic.transform - WARNING - Less than 80% of the genes in PDCD11 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:08:29,303 - pyscenic.transform - WARNING - Less than 80% of the genes in PDLIM5 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:08:29,790 - pyscenic.transform - WARNING - Less than 80% of the genes in PDS5A could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:08:30,737 - pyscenic.transform - WARNING - Less than 80% of the genes in PHF1 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:08:31,207 - pyscenic.transform - WARNING - Less than 80% of the genes in PHF12 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings. S


2024-07-31 12:08:58,079 - pyscenic.transform - WARNING - Less than 80% of the genes in THRB could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:09:01,029 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(4): All regulons derived.

2024-07-31 12:09:01,029 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(4): All regulons derived.

2024-07-31 12:09:01,043 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(4): Done.

2024-07-31 12:09:01,043 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings(4): Done.

2024-07-31 12:09:02,020 - pyscenic.transform - WARNING - Less than 80% of the genes in HBP1 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.

2024-07-31 12:09:

In [18]:
%%bash
pyscenic aucell \
    {rna_loom_ifn} \
    {reg_ofn} \
    --output {scenic_ofn} \
    --auc_threshold 0.15 \
    --num_workers 30


2024-07-31 12:37:58,282 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2024-07-31 12:38:12,382 - pyscenic.cli.pyscenic - INFO - Loading gene signatures.
Create regulons from a dataframe of enriched features.
Additional columns saved: []

2024-07-31 12:38:14,351 - pyscenic.cli.pyscenic - INFO - Calculating cellular enrichment.

2024-07-31 12:38:34,868 - pyscenic.cli.pyscenic - INFO - Writing results to file.


# 3. Creat Regulons(AUC) Assay in Seurat Object

In [ ]:
%%R

library(SCopeLoomR)
library(SCENIC)
library(AUCell)
library(dplyr)

#read loom
loom_fn = "../otp/Tp_scenic.loom"
pyscenic_otp <- SCopeLoomR::open_loom(loom_fn)

# Read AUC information from loom file:
regulonAUC <- SCopeLoomR::get_regulons_AUC(pyscenic_otp, column.attr.name='RegulonsAUC')
regulonAucThresholds <- get_regulon_thresholds(pyscenic_otp)

# correct the data format for 'regulonAucThresholds'
regulonAucThresholds_df <- as.data.frame(regulonAucThresholds)
regulonAucThresholds_correct <- as.character(row.names(regulonAucThresholds_df))
names(regulonAucThresholds_correct) <- regulonAucThresholds_df$regulonAucThresholds
head(as.data.frame(regulonAucThresholds_correct))

# Function for calculating binarized regulon activity
binarizeAUC <- function(auc, thresholds)
{
  thresholds <- thresholds[intersect(names(thresholds), rownames(auc))]
  regulonsCells <- setNames(lapply(names(thresholds), 
                                   function(x) {
                                     trh <- thresholds[x]
                                     names(which(getAUC(auc)[x,]>trh))
                                   }),names(thresholds))
  
  regulonActivity <- reshape2::melt(regulonsCells)
  binaryRegulonActivity <- t(table(regulonActivity[,1], regulonActivity[,2]))
  class(binaryRegulonActivity) <- "matrix"  
  
  return(binaryRegulonActivity)
}


# Binarized regulon activity
binaryRegulonActivity <- binarizeAUC(regulonAUC, regulonAucThresholds_correct)

# Fetch AUC matric 
AUCmat <- AUCell::getAUC(regulonAUC)
AUCmat_df = as.data.frame(AUCmat)
AUCmat_df_z = scale(AUCmat_df)

# Creat AUC and AUCBinary assays 
AUCmat <- AUCell::getAUC(regulonAUC)
Tp[["AUC"]] <- CreateAssayObject(data = AUCmat)
Tp[["AUCBinary"]] <- CreateAssayObject(data = binaryRegulonActivity)
Tp[["AUC"]]@scale.data <- AUCmat_df_z